# Lab 20: Do Foundation Models Beat Classical Forecasts?

*Financial Engineering and the Evidence*, Chapter 20.

**Claim on trial:** pretrained time-series foundation models, used zero-shot, forecast financial and economic series better than classical models.

You will forecast six monthly series one month ahead from 2000 with classical models and with whichever foundation models you have installed, then check significance, contamination and interval calibration. Before each step, write down a prediction.

Foundation models are optional: `pip install chronos-forecasting timesfm` (they need PyTorch).

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import lab20
from fintest.tsfm import load_foundation_models

SOURCE = "download"   # "download" fetches French library and FRED files; "sim" uses fake data to test the code
FILES = {}            # optional: {"ff3": ..., "ff3d": ..., "cpi": ..., "indpro": ..., "unrate": ..., "gs10": ...}
MODELS = ["chronos-bolt-small", "timesfm"]   # "standin" is a test double, not a foundation model
EVAL_START, CUTOFF, CONTEXT = "2000-01", "2024-01", 512
OUT = Path("report")
OUT.mkdir(exist_ok=True)

d = lab20.load(SOURCE, FILES)
SIM = d["simulated"]
fms, msgs = load_foundation_models(MODELS, CONTEXT)
print("series:", list(d["series"]), "| simulated:", SIM)
print("foundation models loaded:", [m.name for m in fms], "| not loaded:", msgs)

## Parts A and B: accuracy and significance

**Predict first:** on which series will a foundation model beat the AR benchmark? Will it beat HAR on realized variance?

In [ ]:
a = lab20.part_a(d, fms, EVAL_START, CUTOFF, CONTEXT)
lab20.fig_heatmap(a, OUT, SIM, EVAL_START)
lab20.fig_cumulative(a, OUT, SIM)
lab20.fig_path(a, OUT, SIM)
display(Image(OUT / "fig20_1_heatmap.png"))
display(Image(OUT / "fig20_2_cumulative.png"))
display(Image(OUT / "fig20_5_path.png"))
a["table"].round(3)

## Part C: contamination

**Predict first:** if a model had seen these series during pretraining, what would its accuracy look like before and after the cutoff?

In [ ]:
c = lab20.part_c(a, CUTOFF)
lab20.fig_contamination(c, OUT, SIM, CUTOFF)
display(Image(OUT / "fig20_3_contamination.png"))
c.round(3)

## Part D: are the intervals honest?

In [ ]:
lab20.fig_coverage(a, OUT, SIM)
display(Image(OUT / "fig20_4_coverage.png"))
a["table"][["series", "model", "coverage_80"]].dropna().round(2)

## Your verdict

1. Does any foundation model beat AR or HAR on any series by a significant margin?
2. Do its gains hold after the cutoff?
3. Are its intervals better calibrated than the classical ones?

Write one paragraph, using the chapter's verdict scale: survives, weakens, fails or provisional.